In [ ]:
# Irish Sign Language (ISL) Recognition Model

This notebook demonstrates how to build a model for recognizing Irish Sign Language (ISL) alphabet using the ISL-HS dataset and MediaPipe for hand landmark detection.

## Setup and Dependencies

First, let's install the necessary libraries:

In [ ]:
!pip install opencv-python mediapipe numpy matplotlib pandas scikit-learn tensorflow

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from google.colab import drive
from tqdm.notebook import tqdm

# Initialize MediaPipe solutions
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

## Mount Google Drive

We'll mount Google Drive to access our dataset:

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Set the path to your dataset
# Update this path to where you've uploaded your ISL-HS dataset
DATASET_PATH = '/content/drive/MyDrive/ISL-HS'

## Data Preparation Functions

Let's define functions to process our images and extract hand landmarks:

In [ ]:
def extract_hand_landmarks(image):
    """
    Extract hand landmarks from an image using MediaPipe.
    
    Args:
        image: Input image
        
    Returns:
        List of landmarks or None if no hand detected
    """
    # Convert the BGR image to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Process the image and detect hands
    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        results = hands.process(image_rgb)
    
    # Check if hand landmarks were detected
    if not results.multi_hand_landmarks:
        return None
    
    # Extract landmarks
    landmarks = []
    for landmark in results.multi_hand_landmarks[0].landmark:
        landmarks.append([landmark.x, landmark.y, landmark.z])
    
    return np.array(landmarks).flatten()

def process_image_folder(folder_path, label):
    """
    Process all images in a folder and extract hand landmarks.
    
    Args:
        folder_path: Path to the folder containing images
        label: Label for the images (letter)
        
    Returns:
        List of (features, label) tuples
    """
    data = []
    
    for filename in os.listdir(folder_path):
        if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
            
        image_path = os.path.join(folder_path, filename)
        image = cv2.imread(image_path)
        
        if image is None:
            print(f"Failed to load image: {image_path}")
            continue
            
        landmarks = extract_hand_landmarks(image)
        
        if landmarks is not None:
            data.append((landmarks, label))
    
    return data

## Load and Process Dataset

Now let's load and process our ISL-HS dataset:

In [ ]:
def load_dataset(dataset_path):
    """
    Load and process the ISL-HS dataset.
    
    Args:
        dataset_path: Path to the dataset
        
    Returns:
        Processed dataset as a list of (features, label) tuples
    """
    all_data = []
    
    # Static letters (A-Y excluding J)
    static_letters = 'ABCDEFGHIKLMNOPQRSTUVWY'
    
    for letter in tqdm(static_letters, desc="Processing static letters"):
        letter_path = os.path.join(dataset_path, letter)
        if os.path.exists(letter_path):
            letter_data = process_image_folder(letter_path, letter)
            all_data.extend(letter_data)
    
    # Note: For dynamic letters (J, X, Z), we would need a different approach
    # This will be implemented separately
    
    return all_data

# Load the dataset
dataset = load_dataset(DATASET_PATH)

## Prepare Data for Training

In [ ]:
# Extract features and labels
X = np.array([item[0] for item in dataset])
y = np.array([item[1] for item in dataset])

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

## Build and Train the Model

In [ ]:
# Build a simple neural network model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(y_categorical.shape[1], activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

## Evaluate the Model

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_accuracy:.4f}")

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## Save the Model

In [ ]:
# Save the model
model.save('/content/drive/MyDrive/isl_recognition_model.h5')

# Save the label encoder
import pickle
with open('/content/drive/MyDrive/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

## Test with a Sample Image

In [ ]:
def predict_letter(image_path):
    """
    Predict the letter from an image.
    
    Args:
        image_path: Path to the image
        
    Returns:
        Predicted letter and confidence
    """
    # Load and process the image
    image = cv2.imread(image_path)
    landmarks = extract_hand_landmarks(image)
    
    if landmarks is None:
        return "No hand detected", 0.0
    
    # Reshape for prediction
    landmarks = landmarks.reshape(1, -1)
    
    # Make prediction
    prediction = model.predict(landmarks)
    predicted_class = np.argmax(prediction)
    confidence = prediction[0][predicted_class]
    
    # Get the letter
    predicted_letter = label_encoder.inverse_transform([predicted_class])[0]
    
    return predicted_letter, confidence

# Test with a sample image
# Replace with a path to your test image
test_image_path = '/content/drive/MyDrive/test_image.jpg'
letter, confidence = predict_letter(test_image_path)
print(f"Predicted letter: {letter}, Confidence: {confidence:.4f}")

# Display the image
image = cv2.imread(test_image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.imshow(image_rgb)
plt.title(f"Predicted: {letter} ({confidence:.4f})")
plt.axis('off')
plt.show()

## Next Steps

1. Implement dynamic gesture recognition for letters J, X, and Z
2. Improve model performance with data augmentation
3. Export the model for use in the FastAPI application
4. Implement real-time recognition with webcam input